In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import importlib, qwen_from_scratch
importlib.reload(qwen_from_scratch)
from qwen_from_scratch import MyQwen

c:\Users\pra19\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# Load Hugging Face model
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # torch_dtype="auto",
    torch_dtype=torch.float32,
    device_map="auto"
)

my_model = MyQwen(hf_model.config).to(hf_model.device)
missing, unexpected = my_model.model.load_state_dict(hf_model.model.state_dict(), strict=True)
my_model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 900.11it/s]


MyQwen(
  (model): QwenModel(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x DecoderLayer(
        (self_attn): AttentionProjections(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        )
        (input_layernorm): RMSNorm()
        (post_attention_layernorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
    (rotary_emb): RotaryEmbedding()
  )
)

# Output Parity Test Naive Greedy

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
input_ids = tokenizer("The capital of United States is", return_tensors="pt").input_ids.to(hf_model.device)

with torch.no_grad():
    hf_out = hf_model(input_ids, output_hidden_states=True)
    my_logits, my_hidden = my_model(input_ids, return_hidden=True)

# Per-layer hidden state comparison.
# hf_out.hidden_states[0] is the embedding output, [i] is after layer i-1.
# NOTE: HF applies the final norm to the LAST entry, so compare only 0..23 here.
for i in range(len(my_hidden) - 1):
    d = (hf_out.hidden_states[i] - my_hidden[i]).abs().max().item()
    print(f"hidden[{i:2d}] max diff = {d:.2e}")

print("HF logits shape:", hf_out.logits.shape)
print("My logits shape:", my_logits.shape)

print("logits max diff =", (hf_out.logits - my_logits).abs().max().item())

# The actual milestone check:
print("M1 parity:", torch.allclose(hf_out.logits, my_logits, atol=1e-4))

# Sanity: both models predict the same next token
print("HF next token :", tokenizer.decode(hf_out.logits[0, -1].argmax()))
print("My next token :", tokenizer.decode(my_logits[0, -1].argmax()))

hidden[ 0] max diff = 0.00e+00
hidden[ 1] max diff = 2.15e-06
hidden[ 2] max diff = 1.91e-06
hidden[ 3] max diff = 3.93e-06
hidden[ 4] max diff = 7.63e-06
hidden[ 5] max diff = 7.63e-06
hidden[ 6] max diff = 7.63e-06
hidden[ 7] max diff = 7.63e-06
hidden[ 8] max diff = 7.63e-06
hidden[ 9] max diff = 7.63e-06
hidden[10] max diff = 7.63e-06
hidden[11] max diff = 7.63e-06
hidden[12] max diff = 7.63e-06
hidden[13] max diff = 7.63e-06
hidden[14] max diff = 7.63e-06
hidden[15] max diff = 9.54e-06
hidden[16] max diff = 9.06e-06
hidden[17] max diff = 1.24e-05
hidden[18] max diff = 1.34e-05
hidden[19] max diff = 1.34e-05
hidden[20] max diff = 2.29e-05
hidden[21] max diff = 3.05e-05
hidden[22] max diff = 3.66e-04
hidden[23] max diff = 1.77e-04
HF logits shape: torch.Size([1, 6, 151936])
My logits shape: torch.Size([1, 6, 151936])
logits max diff = 3.4332275390625e-05
M1 parity: True
HF next token :  Washington
My next token :  Washington
